# Evasion: Impairing Defenses

| Field | Value |
|---|---|
| MITRE ATT&CK | T1562 |
| Tactic | Evasion |
| Difficulty | Intermediate |
| Time | ~30 min |

## Threat Context

In late 2019, the Ryuk ransomware operators refined their playbook with a critical pre-encryption step: systematically disabling security tools before deploying the ransomware payload. Their scripts terminated antivirus processes, disabled Windows Defender real-time protection, stopped logging services, and cleared event logs — all within seconds of gaining administrative access.

The Ryuk group understood that modern EDR and antivirus tools would catch their encryption routines. By killing these defenses first, they created a blind spot — a window where no telemetry was being collected and no behavioral analysis was running. Incident responders arriving after the attack found gaps in logs that made forensic reconstruction significantly harder.

Defense impairment (T1562) is not limited to ransomware. Nation-state actors routinely disable audit logging to cover their tracks, and penetration testers use similar techniques to demonstrate the risk of over-relying on endpoint protection without monitoring for defense tampering.

## What You'll Learn
- How attackers enumerate and disable security tools programmatically
- Common defense impairment techniques: process termination, service manipulation, log clearing
- How to build a defense health monitor that detects when security tools go offline

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Simulates defense impairment detection — does NOT disable real security tools

import subprocess
import os
import json
import time
import platform
from datetime import datetime
from collections import defaultdict

print(f'Platform: {platform.system()} {platform.release()}')
print('All simulations use mock data — no real defenses are impaired.')

## 1. Enumerating Security Tools

Before disabling defenses, attackers first discover what's running. They look for known AV/EDR process names, firewall status, and logging service states. This enumeration phase helps them prioritize which defenses to neutralize.

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Simulate security tool enumeration using mock process lists

# Known security tool process names (used by attackers for enumeration)
SECURITY_PROCESSES = {
    'MsMpEng.exe': {'vendor': 'Microsoft', 'tool': 'Windows Defender', 'type': 'AV'},
    'cb.exe': {'vendor': 'Carbon Black', 'tool': 'CB Defense', 'type': 'EDR'},
    'CylanceSvc.exe': {'vendor': 'Cylance', 'tool': 'CylancePROTECT', 'type': 'EDR'},
    'csfalconservice.exe': {'vendor': 'CrowdStrike', 'tool': 'Falcon', 'type': 'EDR'},
    'SentinelAgent.exe': {'vendor': 'SentinelOne', 'tool': 'Singularity', 'type': 'EDR'},
    'ossec-agent': {'vendor': 'OSSEC', 'tool': 'HIDS Agent', 'type': 'HIDS'},
    'syslog-ng': {'vendor': 'Open Source', 'tool': 'Syslog-NG', 'type': 'Logging'},
    'EventLog': {'vendor': 'Microsoft', 'tool': 'Windows Event Log', 'type': 'Logging'},
}

# Simulated running processes (mock data)
MOCK_RUNNING = [
    {'name': 'MsMpEng.exe', 'pid': 1204, 'status': 'running'},
    {'name': 'EventLog', 'pid': 892, 'status': 'running'},
    {'name': 'csfalconservice.exe', 'pid': 3456, 'status': 'running'},
    {'name': 'syslog-ng', 'pid': 1100, 'status': 'running'},
    {'name': 'chrome.exe', 'pid': 5500, 'status': 'running'},
    {'name': 'explorer.exe', 'pid': 2200, 'status': 'running'},
]

def enumerate_security_tools(process_list, known_tools):
    """Identify running security tools from a process list."""
    found = []
    for proc in process_list:
        if proc['name'] in known_tools:
            info = known_tools[proc['name']].copy()
            info['pid'] = proc['pid']
            info['process'] = proc['name']
            info['status'] = proc['status']
            found.append(info)
    return found

discovered = enumerate_security_tools(MOCK_RUNNING, SECURITY_PROCESSES)

print(f'Security tools discovered: {len(discovered)}\n')
print(f'{"Process":<25s} {"Vendor":<15s} {"Type":<8s} {"PID"}')
print('-' * 60)
for tool in discovered:
    print(f"{tool['process']:<25s} {tool['vendor']:<15s} {tool['type']:<8s} {tool['pid']}")

## 2. Simulating Defense Impairment Techniques

Ryuk operators used batch scripts to kill security processes, disable services, and clear logs. We simulate these actions against our mock process list — no real processes are affected.

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Simulate Ryuk-style defense impairment against MOCK processes

class DefenseImpairmentSimulator:
    """Simulates defense impairment techniques for educational analysis."""
    
    def __init__(self, process_list):
        # Deep copy to avoid mutating original
        self.processes = [p.copy() for p in process_list]
        self.attack_log = []
        self.logs_cleared = False
    
    def kill_process(self, process_name):
        """Simulate terminating a security process."""
        for proc in self.processes:
            if proc['name'] == process_name and proc['status'] == 'running':
                proc['status'] = 'terminated'
                entry = {
                    'time': datetime.now().isoformat(),
                    'action': 'process_kill',
                    'target': process_name,
                    'pid': proc['pid'],
                    'technique': 'T1562.001',
                }
                self.attack_log.append(entry)
                return True
        return False
    
    def disable_service(self, service_name):
        """Simulate disabling a Windows service."""
        entry = {
            'time': datetime.now().isoformat(),
            'action': 'service_disable',
            'target': service_name,
            'command': f'sc config "{service_name}" start= disabled',
            'technique': 'T1562.001',
        }
        self.attack_log.append(entry)
        return True
    
    def clear_event_logs(self):
        """Simulate clearing Windows Event Logs."""
        log_types = ['Security', 'System', 'Application', 'PowerShell']
        for log_type in log_types:
            entry = {
                'time': datetime.now().isoformat(),
                'action': 'log_clear',
                'target': f'{log_type} Event Log',
                'command': f'wevtutil cl {log_type}',
                'technique': 'T1562.002',
            }
            self.attack_log.append(entry)
        self.logs_cleared = True
        return log_types

# Run simulated Ryuk attack sequence
sim = DefenseImpairmentSimulator(MOCK_RUNNING)

# Phase 1: Kill AV/EDR
print('=== Phase 1: Terminate Security Processes ===')
for tool in discovered:
    if tool['type'] in ('AV', 'EDR'):
        result = sim.kill_process(tool['process'])
        status = 'KILLED' if result else 'FAILED'
        print(f'  [{status}] {tool["process"]} (PID {tool["pid"]})')

# Phase 2: Disable services
print('\n=== Phase 2: Disable Services ===')
for svc in ['WinDefend', 'Sense', 'WdNisSvc']:
    sim.disable_service(svc)
    print(f'  [DISABLED] {svc}')

# Phase 3: Clear logs
print('\n=== Phase 3: Clear Event Logs ===')
cleared = sim.clear_event_logs()
for log in cleared:
    print(f'  [CLEARED] {log}')

print(f'\nTotal attack actions logged: {len(sim.attack_log)}')

## 3. Defense Health Monitor

The key defense against T1562 is monitoring the monitors. If a security tool stops running unexpectedly, that event itself should be an alert. This is why defense-in-depth matters — you need an independent system watching for defense impairment.

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Build a defense health monitor

def check_defense_health(current_processes, expected_tools):
    """Compare running processes against expected security tools."""
    running_names = {p['name'] for p in current_processes if p['status'] == 'running'}
    alerts = []
    
    for proc_name, info in expected_tools.items():
        status = 'RUNNING' if proc_name in running_names else 'DOWN'
        if status == 'DOWN':
            alerts.append({
                'alert': f'{info["tool"]} ({proc_name}) is not running',
                'severity': 'CRITICAL' if info['type'] in ('AV', 'EDR') else 'HIGH',
                'vendor': info['vendor'],
                'expected_process': proc_name,
                'recommendation': f'Investigate why {info["tool"]} stopped. Check for T1562.',
            })
    
    return alerts

# Define what SHOULD be running (subset that was in our mock environment)
expected = {
    'MsMpEng.exe': SECURITY_PROCESSES['MsMpEng.exe'],
    'csfalconservice.exe': SECURITY_PROCESSES['csfalconservice.exe'],
    'EventLog': SECURITY_PROCESSES['EventLog'],
    'syslog-ng': SECURITY_PROCESSES['syslog-ng'],
}

# Check AFTER the simulated attack
health_alerts = check_defense_health(sim.processes, expected)

print(f'=== Defense Health Monitor ===')
print(f'Expected tools: {len(expected)}')
print(f'Alerts: {len(health_alerts)}\n')

for alert in health_alerts:
    print(f"[{alert['severity']:8s}] {alert['alert']}")
    print(f"           -> {alert['recommendation']}\n")

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Visualization: Defense impairment timeline

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7))
fig.patch.set_facecolor('#0F0F0F')
fig.suptitle('Defense Impairment Analysis', color='#EDEAE4', fontsize=14, y=1.02)

# Top: Attack action timeline
ax1.set_facecolor('#1A1A1A')
action_types = [e['action'] for e in sim.attack_log]
action_colors = {
    'process_kill': '#CF6C6C',
    'service_disable': '#CF9B6C',
    'log_clear': '#9B6CCF',
}
x_positions = range(len(sim.attack_log))
colors = [action_colors.get(a, '#6C9BCF') for a in action_types]
ax1.bar(x_positions, [1]*len(sim.attack_log), color=colors, width=0.8, edgecolor='#282828')
ax1.set_xticks(list(x_positions))
ax1.set_xticklabels([e.get('target', '')[:12] for e in sim.attack_log],
                     rotation=45, ha='right', fontsize=8)
ax1.set_yticks([])
ax1.set_title('Attack Sequence Timeline', color='#EDEAE4', fontsize=12)
ax1.tick_params(colors='#7A7570')
legend_patches = [mpatches.Patch(color=c, label=l.replace('_', ' ').title())
                  for l, c in action_colors.items()]
ax1.legend(handles=legend_patches, loc='upper right',
           facecolor='#1A1A1A', edgecolor='#282828', labelcolor='#EDEAE4', fontsize=9)
for spine in ax1.spines.values():
    spine.set_edgecolor('#282828')

# Bottom: Defense health status (before vs after)
ax2.set_facecolor('#1A1A1A')
tools = list(expected.keys())
before_status = [1, 1, 1, 1]  # All running before attack
after_status = [0 if any(a['alert'].startswith(expected[t]['tool'])
                for a in health_alerts) else 1 for t in tools]
x = range(len(tools))
width = 0.35
bars1 = ax2.bar([i - width/2 for i in x], before_status, width,
                label='Before Attack', color='#6C9BCF', edgecolor='#282828')
bars2 = ax2.bar([i + width/2 for i in x], after_status, width,
                label='After Attack', color='#CF6C6C', edgecolor='#282828')
ax2.set_xticks(list(x))
ax2.set_xticklabels(tools, fontsize=9)
ax2.set_yticks([0, 1])
ax2.set_yticklabels(['Down', 'Running'])
ax2.set_title('Security Tool Status', color='#EDEAE4', fontsize=12)
ax2.tick_params(colors='#7A7570')
ax2.legend(facecolor='#1A1A1A', edgecolor='#282828', labelcolor='#EDEAE4', fontsize=9)
for spine in ax2.spines.values():
    spine.set_edgecolor('#282828')

plt.tight_layout()
plt.show()

## Defender's Perspective

| Indicator | Type | Detection |
|---|---|---|
| Security process terminated unexpectedly | Process Event | EDR watchdog / heartbeat monitoring |
| Service start type changed to "disabled" | Registry Change | Windows Event ID 7040 (Service Control Manager) |
| Event logs cleared | Log Event | Windows Event ID 1102 (Audit Log Cleared) — this event survives the clear |
| Tamper protection alerts from EDR | Alert | Central EDR console — monitor for self-protection events |

## Article Seed

**Title:** Why Ryuk Kills Your Antivirus Before Encrypting Your Files

**Opening:** Ransomware doesn't just encrypt — it disarms first. The Ryuk operators perfected a playbook where the first action after gaining admin access was systematically killing every security tool on the system. Understanding defense impairment (T1562) is critical for anyone designing defense-in-depth strategies.

**Sections:**
1. The Ryuk pre-encryption killchain
2. How attackers enumerate and disable security tools
3. Building a defense health monitor that watches the watchers
4. Why tamper protection and out-of-band monitoring are essential

**Tags:** cybersecurity, ransomware, evasion, defense-in-depth, python

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Self-check assertions

# Verify enumeration found the right tools
assert len(discovered) == 4, f'Expected 4 security tools, got {len(discovered)}'

# Verify attack simulation logged all actions
assert len(sim.attack_log) >= 7, 'Should have at least 7 attack actions logged'

# Verify killed processes are marked as terminated
killed = [p for p in sim.processes if p['status'] == 'terminated']
assert len(killed) == 2, 'Should have terminated 2 AV/EDR processes'

# Verify health monitor detected the kills
assert len(health_alerts) >= 2, 'Health monitor should raise at least 2 alerts'
critical_alerts = [a for a in health_alerts if a['severity'] == 'CRITICAL']
assert len(critical_alerts) >= 1, 'Should have at least 1 CRITICAL alert'

# Verify logs were marked as cleared
assert sim.logs_cleared is True, 'Log clearing should be recorded'

print('All assertions passed.')